# 04 — Solution-augmented BKT baseline

This notebook retrains the effective NumPy BKT baseline from notebook 03 on
the reformatted MathDial data created in notebook 00. Every dialogue now starts
with a synthetic `solution` unit representing the student's known incorrect
solution:

- `correct=False`;
- its KCs are the ordered union of KCs annotated anywhere in the dialogue.

The model is therefore a **retrospective solution-KC baseline**. The solution
union exposes which KCs occur later in the dialogue, so this is not a causal
real-time reproduction of the paper. All other preparation, model, aggregation,
fallback, and metric decisions remain aligned with notebook 03.


## 1. Data and evaluation contract

Preparation is performed in this order:

1. Load `data/misconception/mathdial_train.csv` and `mathdial_test.csv`.
2. Retain typical-confusion and typical-interaction scores of at least 1.
3. Remove failed ATC annotations.
4. Treat turns without both a correctness label and KCs as untagged.
5. Apply the paper's final-turn override from `self-correctness`.
6. Retain dialogues with at least three tagged units: the solution plus at
   least two usable real turns.
7. Expand every tagged unit into one pseudo-observation per KC.

The solution is **included in inference and state updating but never scored**.
Two turn-level evaluations are reported:

| Evaluation | State updates retained | Scored turns |
| --- | --- | --- |
| All real turns | solution, then preceding real turns | every usable real turn |
| Paper-matched | solution and first usable real turn | second and later usable real turns |

Paper-matched evaluation removes the solution and first real turn only from
the metric calculation. Both remain in the BKT sequence and update mastery.


In [1]:
import os
import sys
import json
import re
import time
from ast import literal_eval
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

_here = Path.cwd()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "data" / "misconception").exists():
        os.chdir(_candidate)
        break
else:
    raise FileNotFoundError(f"Could not find data/misconception above {_here}")

sys.path.insert(0, str(Path.cwd() / "extension"))
from scripts import bkt

DATA = Path("data/misconception")
MODELS = Path("extension/models")
RESULTS = Path("extension/results")
MODELS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

def parse_kcs(value):
    """Parse a CSV KC cell; blanks and malformed values become an empty list."""
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = literal_eval(text)
    except (SyntaxError, ValueError):
        return []
    return parsed if isinstance(parsed, list) else []

def parse_correct(value):
    """Parse True/False while preserving blank correctness as None."""
    text = str(value).strip().lower()
    if text == "true":
        return True
    if text == "false":
        return False
    return None

def read_reformatted(split):
    return pd.read_csv(
        DATA / f"mathdial_{split}.csv",
        keep_default_na=False,
        converters={"kcs": parse_kcs},
    )

train_raw = read_reformatted("train")
test_raw = read_reformatted("test")
print("working directory:", Path.cwd())
print("raw dialogues — train:", train_raw.dialogue_id.nunique(),
      "test:", test_raw.dialogue_id.nunique())


working directory: /Users/tandon.utsav2/Desktop/Experiment_1
raw dialogues — train: 2253 test: 595


## 2. Prepare the solution-augmented sequences

A tagged **unit** is counted once even when it contains several KCs. Only after
the three-unit rule passes is the unit expanded into KC pseudo-observations.
The real-turn rank counts usable tagged turns, not raw turn numbers, matching
the paper's exclusion of the first tagged turn.


In [2]:
def prepare_solution_long(raw, split):
    typical_mask = (
        pd.to_numeric(raw["self-typical-confusion"]) >= 1
    ) & (
        pd.to_numeric(raw["self-typical-interactions"]) >= 1
    )
    typical = raw.loc[typical_mask].copy()

    rows = []
    n_failed = n_short = n_kept = 0
    n_real_tagged = n_solution_pseudo = 0

    for dialogue_id, group in typical.groupby("dialogue_id", sort=False):
        group = group.reset_index(drop=True)
        assert group.iloc[0]["turn"] == "solution"
        assert (group["turn"] == "solution").sum() == 1

        solution_kcs = group.iloc[0]["kcs"]
        solution_correct = parse_correct(group.iloc[0]["correct"])
        assert solution_correct is False

        real = []
        for _, record in group.iloc[1:].iterrows():
            match = re.search(r"\d+", str(record["turn"]))
            if not match:
                continue
            kcs = record["kcs"]
            correct = parse_correct(record["correct"])
            if not kcs:
                correct = None
            real.append({
                "turn": int(match.group()),
                "correct": correct,
                "kcs": kcs,
            })

        # Failed ATC records have no correctness or KCs on any real turn.
        if not any(item["correct"] is not None or item["kcs"] for item in real):
            n_failed += 1
            continue

        # Paper's final-actual-turn correctness override.
        if real and real[-1]["kcs"] and real[-1]["correct"] is not None:
            outcome = str(group.iloc[-1]["self-correctness"])
            if outcome == "Yes":
                real[-1]["correct"] = True
            elif outcome == "No":
                real[-1]["correct"] = False
            elif outcome == "Yes, but I had to reveal the answer":
                real[-1]["correct"] = None

        tagged_real = [
            item for item in real
            if item["correct"] is not None and item["kcs"]
        ]
        tagged_units = int(bool(solution_kcs)) + len(tagged_real)
        if tagged_units < 3:
            n_short += 1
            continue

        n_kept += 1
        n_real_tagged += len(tagged_real)
        n_solution_pseudo += len(solution_kcs)

        for kc in solution_kcs:
            rows.append({
                "dialogue_idx": int(dialogue_id),
                "turn": 0,
                "unit": "solution",
                "real_rank": 0,
                "correct": 0,
                "kc": str(kc),
            })

        for real_rank, item in enumerate(tagged_real, start=1):
            for kc in item["kcs"]:
                rows.append({
                    "dialogue_idx": int(dialogue_id),
                    "turn": item["turn"],
                    "unit": f"turn {item['turn']}",
                    "real_rank": real_rank,
                    "correct": int(item["correct"]),
                    "kc": str(kc),
                })

    long_df = pd.DataFrame(rows)
    audit = {
        "split": split,
        "raw_dialogues": raw.dialogue_id.nunique(),
        "after_typical": typical.dialogue_id.nunique(),
        "failed_removed": n_failed,
        "fewer_than_3_tagged_removed": n_short,
        "dialogues_kept": n_kept,
        "tagged_units_including_solution": n_real_tagged + n_kept,
        "real_tagged_turns": n_real_tagged,
        "solution_pseudo_observations": n_solution_pseudo,
        "pseudo_observations": len(long_df),
        "distinct_kcs": long_df["kc"].nunique(),
        "all_real_scored_turns": n_real_tagged,
        "paper_matched_scored_turns": n_real_tagged - n_kept,
    }
    return long_df, audit

train_long, train_audit = prepare_solution_long(train_raw, "train")
test_long, test_audit = prepare_solution_long(test_raw, "test")
audit = pd.DataFrame([train_audit, test_audit]).set_index("split")
print(audit.to_string())


       raw_dialogues  after_typical  failed_removed  fewer_than_3_tagged_removed  dialogues_kept  tagged_units_including_solution  real_tagged_turns  solution_pseudo_observations  pseudo_observations  distinct_kcs  all_real_scored_turns  paper_matched_scored_turns
split                                                                                                                                                                                                                                                                   
train           2253           2253              18                          185            2050                            12498              10448                          9268                33221           142                  10448                        8398
test             595            595               7                           73             515                             3015               2500                          2304                 8092      

### Population parity with notebook 03

The solution plus two-real-turn rule must preserve notebook 03's retained
population: 2,050 training and 515 test dialogues. Failed annotations remain
a separate provenance removal, while the new short-dialogue rule removes the
same 185/73 otherwise-usable dialogues as before.


In [3]:
expected = {
    "train": {"failed_removed": 18, "fewer_than_3_tagged_removed": 185,
              "dialogues_kept": 2050, "real_tagged_turns": 10448,
              "all_real_scored_turns": 10448,
              "paper_matched_scored_turns": 8398},
    "test": {"failed_removed": 7, "fewer_than_3_tagged_removed": 73,
             "dialogues_kept": 515, "real_tagged_turns": 2500,
             "all_real_scored_turns": 2500,
             "paper_matched_scored_turns": 1985},
}
for split, checks in expected.items():
    for field, value in checks.items():
        assert int(audit.loc[split, field]) == value, (
            split, field, audit.loc[split, field], value
        )

for frame in [train_long, test_long]:
    assert not frame[["correct", "kc"]].isna().any().any()
    assert set(frame["correct"].unique()) <= {0, 1}
    per_dialogue = frame.drop_duplicates(["dialogue_idx", "real_rank"])
    assert per_dialogue.groupby("dialogue_idx").size().min() >= 3
    assert per_dialogue.groupby("dialogue_idx")["real_rank"].min().eq(0).all()

population_stats = pd.DataFrame({
    "original_notebook_03": [23953, 5788],
    "solution_augmented": [len(train_long), len(test_long)],
    "added_solution_rows": [
        train_audit["solution_pseudo_observations"],
        test_audit["solution_pseudo_observations"],
    ],
}, index=["train", "test"])
print("population assertions passed\n")
print(population_stats.to_string())
print(f"\ntraining pseudo-observation correctness rate: "
      f"{train_long['correct'].mean():.4f}")


population assertions passed

       original_notebook_03  solution_augmented  added_solution_rows
train                 23953               33221                 9268
test                   5788                8092                 2304

training pseudo-observation correctness rate: 0.3581


## 3. Fit the effective BKT model

The model is unchanged from notebook 03: standard four-parameter,
no-forgetting BKT; five deterministic EM restarts; guess/slip identifiability
bounds; and an observation-weighted parameter fallback learned from
nondegenerate KCs. The only change is the added incorrect solution evidence.


In [4]:
kc_stats = train_long.groupby("kc")["correct"].agg(n="count", n_correct="sum")
degenerate_mask = (
    (kc_stats["n_correct"] == 0)
    | (kc_stats["n_correct"] == kc_stats["n"])
)
degenerate = kc_stats[degenerate_mask]
nondegenerate = kc_stats[~degenerate_mask]

print("training KCs:", len(kc_stats))
print("fitted individually:", len(nondegenerate))
print("assigned fallback:", len(degenerate))
print("degenerate pseudo-observations:", int(degenerate["n"].sum()))

FIT_CONFIG = {"n_restarts": 5, "max_iter": 100, "seed": 221}
started = time.time()
fitted = bkt.fit_bkt(train_long, **FIT_CONFIG)
fit_seconds = time.time() - started
print(f"fit time: {fit_seconds:.1f}s")
print("fallback:", {name: round(value, 4)
                    for name, value in fitted.fallback.items()})


training KCs: 142
fitted individually: 131
assigned fallback: 11
degenerate pseudo-observations: 23


[fit_bkt] 11 degenerate KCs assigned the observation-weighted fallback from nondegenerate KCs.
fit time: 101.8s
fallback: {'prior': 0.001, 'learns': 0.8892, 'guesses': 0.0022, 'slips': 0.4535}


## 4. Evaluate all real turns and the paper-matched subset

Predictions are generated before observing the current response. KC-level
probabilities are averaged arithmetically to one probability per true turn.
Accuracy uses the paper's `np.round` hard predictions and F1 treats correct
as the positive class.

For a controlled comparison, the saved effective baseline from notebook 03 is
reloaded and evaluated on the identical real-turn rows. Thus each reported
difference changes the fitted model and solution history, not the targets.


In [5]:
def aggregate_real_turns(pseudo_predictions, prediction_name):
    real = pseudo_predictions[pseudo_predictions["unit"] != "solution"].copy()
    label_counts = real.groupby(["dialogue_idx", "turn"])["correct"].nunique()
    assert label_counts.eq(1).all()
    return (
        real.groupby(["dialogue_idx", "turn", "unit", "real_rank"], sort=False)
        .agg(**{prediction_name: ("pred", "mean")}, correct=("correct", "first"))
        .reset_index()
    )

def score_turns(turns, prediction_column, minimum_real_rank):
    scored = turns[turns["real_rank"] >= minimum_real_rank]
    labels = scored["correct"].to_numpy(dtype=int)
    probabilities = scored[prediction_column].to_numpy(dtype=float)
    hard = np.round(probabilities).astype(int)
    return {
        "n": len(scored),
        "accuracy": accuracy_score(labels, hard),
        "auc": roc_auc_score(labels, probabilities),
        "f1": f1_score(labels, hard, average="binary", pos_label=1,
                       zero_division=0),
    }

# Solution-augmented predictions: S0 and R1 update the state before R2+.
new_pseudo_predictions = fitted.predict_long(test_long)
new_turns = aggregate_real_turns(new_pseudo_predictions, "solution_bkt_pred")

# Notebook-03 control: same real rows, without solution observations.
with (MODELS / "bkt_original.json").open() as handle:
    original_payload = json.load(handle)
original_fitted = bkt.FittedBKT(
    original_payload["per_skill"], original_payload["fallback"]
)
original_test_long = test_long[test_long["unit"] != "solution"].copy()
original_pseudo_predictions = original_fitted.predict_long(original_test_long)
original_turns = aggregate_real_turns(
    original_pseudo_predictions, "original_bkt_pred"
)

turn_predictions = original_turns.merge(
    new_turns,
    on=["dialogue_idx", "turn", "unit", "real_rank", "correct"],
    how="inner",
    validate="one_to_one",
)
assert len(turn_predictions) == 2500

protocols = {
    "all_real_turns": 1,
    "paper_matched_R2_plus": 2,
}
metric_rows = []
for protocol, minimum_rank in protocols.items():
    for model, column in [
        ("original_bkt", "original_bkt_pred"),
        ("solution_augmented_bkt", "solution_bkt_pred"),
    ]:
        metric_rows.append({
            "protocol": protocol,
            "model": model,
            **score_turns(turn_predictions, column, minimum_rank),
        })

metrics = pd.DataFrame(metric_rows)
print(metrics.set_index(["protocol", "model"]).round(4).to_string())

deltas = (
    metrics.pivot(index="protocol", columns="model",
                  values=["accuracy", "auc", "f1"])
)
delta_rows = {}
for protocol in protocols:
    delta_rows[protocol] = {
        metric: (
            deltas.loc[protocol, (metric, "solution_augmented_bkt")]
            - deltas.loc[protocol, (metric, "original_bkt")]
        )
        for metric in ["accuracy", "auc", "f1"]
    }
metric_deltas = pd.DataFrame(delta_rows).T
print("\nsolution-augmented minus original")
print(metric_deltas.round(4).to_string())

# Confirm that reloading notebook 03 reproduces its saved reference metrics.
reference = pd.read_csv(RESULTS / "original_bkt_reference.csv")
reference_lookup = {
    "all_real_turns": reference[reference["model"].str.contains("all turns")].iloc[0],
    "paper_matched_R2_plus": reference[
        reference["model"].str.contains("paper-aligned")
    ].iloc[0],
}
for protocol, row in reference_lookup.items():
    reproduced = metrics[
        (metrics["protocol"] == protocol)
        & (metrics["model"] == "original_bkt")
    ].iloc[0]
    for metric in ["accuracy", "auc", "f1"]:
        assert np.isclose(reproduced[metric], row[metric], atol=1e-12)
print("\nnotebook-03 reference metrics reproduced exactly")


                                                 n  accuracy     auc      f1
protocol              model                                                 
all_real_turns        original_bkt            2500    0.5980  0.6278  0.5571
                      solution_augmented_bkt  2500    0.5312  0.5843  0.5936
paper_matched_R2_plus original_bkt            1985    0.6050  0.6402  0.5556
                      solution_augmented_bkt  1985    0.5088  0.5640  0.6064

solution-augmented minus original
                       accuracy     auc      f1
all_real_turns          -0.0668 -0.0435  0.0365
paper_matched_R2_plus   -0.0962 -0.0762  0.0508

notebook-03 reference metrics reproduced exactly


### Result interpretation

The solution-augmented model is worse on the two threshold-independent/core
headline measures in both evaluations:

- **All real turns:** accuracy falls from 0.5980 to 0.5312 and AUC from
  0.6278 to 0.5843; F1 rises from 0.5571 to 0.5936.
- **Paper-matched:** accuracy falls from 0.6050 to 0.5088 and AUC from
  0.6402 to 0.5640; F1 rises from 0.5556 to 0.6064.

The F1 increase does not indicate better ranking. The augmented model predicts
the positive (`correct`) class for 69.4% of all-real targets and 77.4% of
paper-matched targets, compared with 44.8% and 41.5% for the original model.
This raises recall while sharply reducing accuracy and discrimination.

The parameter fit explains the shift: all fitted KC priors reach the lower
0.001 bound, reflecting the forced incorrect observation attached to every KC
at the start of each dialogue. Learning estimates then become high, with a
median of 0.730. The result is therefore evidence that treating the full KC
union as simultaneously attempted and incorrect is too strong for standard
BKT, not evidence that the initial solution contains no useful information.

### Fallback and parameter diagnostics

Adding one incorrect observation for every dialogue-level KC changes class
balance and can turn formerly all-correct KCs into estimable KCs. The following
audit distinguishes the requested evaluation results from fallback coverage
and boundary-fitting diagnostics.


In [6]:
train_kcs = set(train_long["kc"])
test_kcs = set(test_long["kc"])
fallback_kcs = set(degenerate.index.astype(str)) | (test_kcs - train_kcs)
fallback_test_rows = int(test_long["kc"].isin(fallback_kcs).sum())
fallback_scored_real_rows = int(
    test_long[
        (test_long["unit"] != "solution")
        & test_long["kc"].isin(fallback_kcs)
    ].shape[0]
)

params = pd.DataFrame(fitted.per_skill).T[list(bkt.PARAM_NAMES)]
at_boundary = (
    np.isclose(params["prior"], 0.001)
    | np.isclose(params["prior"], 0.999)
    | np.isclose(params["learns"], 0.001)
    | np.isclose(params["learns"], 0.999)
    | np.isclose(params["guesses"], 0.001)
    | np.isclose(params["guesses"], 0.49)
    | np.isclose(params["slips"], 0.001)
    | np.isclose(params["slips"], 0.49)
)

print("fallback KCs:", len(fallback_kcs))
print("fallback test pseudo-rows (all units):", fallback_test_rows)
print("fallback test pseudo-rows (scored real units):",
      fallback_scored_real_rows)
print("KCs with at least one boundary parameter:", int(at_boundary.sum()))
print("\nparameter summary:")
print(params.describe().round(3).to_string())


fallback KCs: 16
fallback test pseudo-rows (all units): 32
fallback test pseudo-rows (scored real units): 21
KCs with at least one boundary parameter: 142

parameter summary:
         prior   learns  guesses    slips
count  142.000  142.000  142.000  142.000
mean     0.001    0.727    0.013    0.316
std      0.000    0.273    0.050    0.191
min      0.001    0.001    0.001    0.001
25%      0.001    0.538    0.001    0.145
50%      0.001    0.730    0.001    0.416
75%      0.001    0.999    0.001    0.490
max      0.001    0.999    0.312    0.490


## 5. Save and reload the baseline

The saved model records the retrospective solution-union contract and filter
threshold. Turn-level predictions are saved with both original and augmented
probabilities so every metric row can be reconstructed without refitting.


In [7]:
model_path = MODELS / "bkt_solution_baseline.json"
metrics_path = RESULTS / "bkt_solution_baseline_metrics.csv"
predictions_path = RESULTS / "bkt_solution_baseline_turn_predictions.csv"

payload = {
    "model_type": "BKT with retrospective incorrect-solution KC union",
    "param_names": list(bkt.PARAM_NAMES),
    "per_skill": fitted.per_skill,
    "fallback": fitted.fallback,
    "metadata": {
        "saved_utc": datetime.now(timezone.utc).isoformat(),
        "fit": FIT_CONFIG,
        "fit_seconds": fit_seconds,
        "train_pseudo_observations": len(train_long),
        "test_pseudo_observations": len(test_long),
        "train_dialogues": train_audit["dialogues_kept"],
        "test_dialogues": test_audit["dialogues_kept"],
        "n_degenerate_kcs": len(degenerate),
        "solution_contract": (
            "correct=False; ordered union of all dialogue KCs; included in "
            "BKT state updates; excluded from every metric"
        ),
        "data_contract": (
            "typical cutoff 1; failed annotations removed; final-turn override; "
            "correctness=None excluded; minimum three tagged units"
        ),
        "fallback_policy": (
            "complete observation-weighted parameter vector from "
            "nondegenerate KCs; used for degenerate/unseen KCs"
        ),
    },
}
with model_path.open("w") as handle:
    json.dump(payload, handle, indent=2)
metrics.to_csv(metrics_path, index=False)

turn_predictions = turn_predictions.copy()
turn_predictions["in_all_real_evaluation"] = True
turn_predictions["in_paper_matched_evaluation"] = (
    turn_predictions["real_rank"] >= 2
)
turn_predictions.to_csv(predictions_path, index=False)

with model_path.open() as handle:
    loaded = json.load(handle)
reloaded = bkt.FittedBKT(loaded["per_skill"], loaded["fallback"])
reloaded_predictions = reloaded.predict_long(test_long)["pred"].to_numpy()
max_difference = float(np.max(np.abs(
    reloaded_predictions - new_pseudo_predictions["pred"].to_numpy()
)))
assert max_difference < 1e-12

print("saved:", model_path)
print("saved:", metrics_path)
print("saved:", predictions_path)
print(f"reload max prediction difference: {max_difference:.2e}")


saved: extension/models/bkt_solution_baseline.json
saved: extension/results/bkt_solution_baseline_metrics.csv
saved: extension/results/bkt_solution_baseline_turn_predictions.csv
reload max prediction difference: 0.00e+00


## Summary

- The three-tagged-unit rule preserves notebook 03's dialogue population while
  adding one incorrect solution unit to every retained sequence.
- All real-turn evaluation scores every usable real response after updating on
  the solution.
- Paper-matched evaluation scores only the second and later usable real turns;
  both the solution and first real response remain state updates.
- Original and augmented models are compared on identical targets using the
  paper's turn aggregation, accuracy, AUC, and binary-F1 definitions.
- The model is retrospective because solution KCs are derived from the full
  dialogue; its results must not be interpreted as causal online performance.
